# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [78]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr


In [79]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [80]:
# Load tables
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')

In [81]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{' g__Reyranella', ' g__Mycobacterium', ' g__Janibacter'}

Filtered Unique to V4 (>=10% prevalence):
{' g__Lactococcus', ' g__Pantoea', ' g__Chryseobacterium', ' g__Dolosigranulum', ' g__Massilia', ' g__Aeromonas', ' g__Jeotgalicoccus', ' g__Frederiksenia', ' g__Empedobacter', ' g__Brachybacterium', ' g__Moraxella', ' g__Turicella', ' g__Marinomonas', ' g__Gardnerella', ' g__Bradyrhizobium', ' g__Capnocytophaga', ' g__Leptotrichia', ' g__Finegoldia', ' g__Peptoniphilus', ' g__Atopobium', ' g__Abiotrophia', ' g__Fusobacterium', ' g__Blautia', ' g__Peptostreptococcus', ' g__Leuconostoc', ' g__Psychrobacter', ' g__Fenollaria', ' g__Luteimonas', ' g__Aggregatibacter', ' g__Vibrio', ' g__Aerococcus', ' g__Alloiococcus', ' g__Geobacillus', ' g__Bifidobacterium', ' g__Enhydrobacter', ' g__Stenotrophomonas'}

Filtered Shared Taxa (>=10% prevalence):
{' g__Lysobacter', ' g__Sphingopyxis', ' g__Brevundimonas', ' g__Cutibacterium', ' g__Brevibacterium',

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_19132/4290181383.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [82]:
# Read and transform V1-V3 and V4 Genus level absolute abundance table
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')

In [83]:
# Extract the row corresponding to 'g__Cutibacterium' for V1-V3
V1_V3_cutibacterium_df = V1V3_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V1_V3_cutibacterium_df.index = [' g__Cutibacterium V1-V3']

# Transpose the DataFrame
V1_V3_cutibacterium_df = V1_V3_cutibacterium_df.T

# Display the transformed DataFrame
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3
LAMI.RD001.D0.C1,0.626691
LAMI.RD001.D0.C2,0.479703
LAMI.RD001.D14.C1,0.592730
LAMI.RD001.D14.C2,0.467233
LAMI.RD001.D28.C1,0.628018
...,...
LAMI.RD319.D14.C1,0.504112
LAMI.RD319.D21.C3,0.266118
LAMI.RD319.D14.C2,0.614221
LAMI.RD319.D9.C1,0.326347


In [84]:
# Extract the row corresponding to 'g__Cutibacterium' for V4
V4_cutibacterium_df = V4_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V4_cutibacterium_df.index = [' g__Cutibacterium V4']

# Transpose the DataFrame
V4_cutibacterium_df = V4_cutibacterium_df.T

# Display the transformed DataFrame
V4_cutibacterium_df

,g__Cutibacterium V4
LAMI.RD001.D0.C1,0.000265
LAMI.RD001.D14.C1,0.000000
LAMI.RD004.D0.C2,0.000265
LAMI.RD001.D0.C2,0.000796
LAMI.RD004.D28.C1,0.000000
...,...
LAMI.RD319.D16.C2,0.000796
LAMI.RD319.D28.C1,0.000000
LAMI.RD318.D9.C2,0.002653
LAMI.RD319.D28.C2,0.000000


In [85]:
# Get ranks for V1-V3
V1_V3_cutibacterium_df['V1-V3'] = V1_V3_cutibacterium_df[' g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD001.D0.C1,0.626691,31
LAMI.RD001.D0.C2,0.479703,12
LAMI.RD001.D14.C1,0.592730,25
LAMI.RD001.D14.C2,0.467233,11
LAMI.RD001.D28.C1,0.628018,33
...,...,...
LAMI.RD319.D14.C1,0.504112,15
LAMI.RD319.D21.C3,0.266118,4
LAMI.RD319.D14.C2,0.614221,29
LAMI.RD319.D9.C1,0.326347,5


In [86]:
# Get ranks for V4
V4_cutibacterium_df['V4'] = V4_cutibacterium_df[' g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df

,g__Cutibacterium V4,V4
LAMI.RD001.D0.C1,0.000265,12
LAMI.RD001.D14.C1,0.000000,1
LAMI.RD004.D0.C2,0.000265,12
LAMI.RD001.D0.C2,0.000796,42
LAMI.RD004.D28.C1,0.000000,1
...,...,...
LAMI.RD319.D16.C2,0.000796,42
LAMI.RD319.D28.C1,0.000000,1
LAMI.RD318.D9.C2,0.002653,79
LAMI.RD319.D28.C2,0.000000,1


In [87]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
v1_v3_column = V1_V3_cutibacterium_df['V1-V3']  # Adjust column name if needed
v4_column = V4_cutibacterium_df['V4']  # Adjust column name if needed

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column], axis=1)

# Rename columns for clarity (optional)
combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4
LAMI.RD001.D0.C1,31.0,12.0
LAMI.RD001.D0.C2,12.0,42.0
LAMI.RD001.D14.C1,25.0,1.0
LAMI.RD001.D14.C2,11.0,1.0
LAMI.RD001.D28.C1,33.0,12.0
...,...,...
LAMI.RD318.D28.C3,77.0,51.0
LAMI.RD319.D14.C1,15.0,12.0
LAMI.RD319.D14.C2,29.0,1.0
LAMI.RD319.D9.C1,5.0,1.0


In [88]:
# Scatter plot
plt.figure(figsize=(6, 6))
plt.scatter(
    combined_cutibacterium_df.iloc[:, 0], 
    combined_cutibacterium_df.iloc[:, 1], 
    color='#ffa505', 
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7, 
    label='Samples'
)

# Extract x and y data
x = combined_cutibacterium_df.iloc[:, 0]
y = combined_cutibacterium_df.iloc[:, 1]

# Compute the best-fit line
m, b = np.polyfit(x, y, 1)

# Compute Pearson correlation coefficient and p-value
r, p_value = pearsonr(x, y)

# Plot the best-fit line with the correlation and p-value in the label
plt.plot(x, m * x + b, color='darkorange', label=f'Pearson Correlation: {r:.2f}\np-value: {p_value:.3g}')

# Title and labels
plt.title('Cutibacterium by Primer Set', fontsize=16)
plt.xlabel('Ranked Relative Abundance (V1-V3)', fontsize=14)
plt.ylabel('Ranked Relative Abundance (V4)', fontsize=14)

# Grid, legend, and save
plt.grid(True)
plt.legend()
plt.savefig('../Figures/16S_Figures/primer_comparison/Cutbacterium_V1V3_vs_V4_correlation_ranks.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_19132/2837332338.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Comparison of 16S V1-V3, 16S V4, and shotgun of Cutibacterium per-sample ranked relative abundances with intersectional samples

In [89]:
# Load tables
veba_slcs = pd.read_csv('../Data/metaG/VEBA_analysis/output_files/X_mags.with_taxonomy.with_slcs_name.csv')
veba_slcs_subset = veba_slcs.iloc[:, 4:]
veba_slcs_subset = veba_slcs_subset.set_index('name')
veba_slcs_subset.index.name = None

veba_slcs_subset_collapsed = veba_slcs_subset.groupby(veba_slcs_subset.index).sum()
relative_abundance_df = veba_slcs_subset_collapsed.div(veba_slcs_subset_collapsed.sum(axis=0), axis=1)
relative_abundance_df = relative_abundance_df.T
cutibacterium_only = relative_abundance_df[['Cutibacterium acnes']]
cutibacterium_only = cutibacterium_only.rename(columns={'Cutibacterium acnes': 'Cutibacterium acnes Shotgun'})

cutibacterium_only.index = cutibacterium_only.index.str.split('_').str[:4].str.join('_')

cutibacterium_only.index = cutibacterium_only.index.str.replace('_', '.', regex=False)


# Get ranks for shotgun
cutibacterium_only['Shotgun'] = cutibacterium_only['Cutibacterium acnes Shotgun'].rank(method='min').astype(int)
cutibacterium_only

,Cutibacterium acnes Shotgun,Shotgun
LAMI.RD308.D9.C3,0.975484,20
LAMI.RD306.D28.C3,0.802056,4
LAMI.RD310.D16.C2,0.925635,10
LAMI.RD304.D14.C3,0.948995,14
LAMI.RD304.D11.C1,0.943899,12
LAMI.RD306.D14.C3,0.790596,3
LAMI.RD308.D0.C3,0.989375,23
LAMI.RD306.D23.C1,0.961753,15
LAMI.RD308.D23.C2,0.874240,7
LAMI.RD310.D7.C3,0.975159,19


In [90]:
# Get ranks for V1-V3 for samples matching with shotgun
V1_V3_cutibacterium_df_shotgun_subset = V1_V3_cutibacterium_df.loc[V1_V3_cutibacterium_df.index.isin(cutibacterium_only.index)]
V1_V3_cutibacterium_df_shotgun_subset = V1_V3_cutibacterium_df_shotgun_subset[[' g__Cutibacterium V1-V3']]

# Get ranks for subsetted V1-V3
V1_V3_cutibacterium_df_shotgun_subset['V1-V3'] = V1_V3_cutibacterium_df_shotgun_subset[' g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df_shotgun_subset

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD304.D14.C3,0.892014,7
LAMI.RD304.D7.C1,0.895994,8
LAMI.RD304.D0.C1,0.904484,9
LAMI.RD306.D0.C3,0.856991,4
LAMI.RD306.D11.C1,0.863094,5
LAMI.RD308.D0.C2,0.996020,24
LAMI.RD306.D14.C1,0.738392,2
LAMI.RD304.D0.C3,0.909790,10
LAMI.RD308.D0.C3,0.993632,21
LAMI.RD306.D23.C1,0.942425,13


In [91]:
# Get ranks for V4 for samples matching with shotgun
V4_cutibacterium_df_shotgun_subset = V4_cutibacterium_df.loc[V4_cutibacterium_df.index.isin(cutibacterium_only.index)]
V4_cutibacterium_df_shotgun_subset = V4_cutibacterium_df_shotgun_subset[[' g__Cutibacterium V4']]

# Get ranks for subsetted V4
V4_cutibacterium_df_shotgun_subset['V4'] = V4_cutibacterium_df_shotgun_subset[' g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df_shotgun_subset

,g__Cutibacterium V4,V4
LAMI.RD304.D0.C1,0.004245,11
LAMI.RD304.D0.C3,0.003184,6
LAMI.RD304.D11.C1,0.003980,10
LAMI.RD306.D23.C1,0.005306,13
LAMI.RD308.D0.C2,0.024675,19
LAMI.RD306.D11.C1,0.000531,2
LAMI.RD308.D0.C3,0.024410,18
LAMI.RD306.D14.C1,0.001857,4
LAMI.RD304.D14.C3,0.005572,14
LAMI.RD308.D14.C3,0.035553,22


In [92]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
shotgun_column = cutibacterium_only['Shotgun']
v1_v3_column = V1_V3_cutibacterium_df_shotgun_subset['V1-V3']
v4_column = V4_cutibacterium_df_shotgun_subset['V4']

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column, shotgun_column], axis=1)

# Rename columns for clarity (optional)
# combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4,Shotgun
LAMI.RD304.D14.C3,7,14.0,14
LAMI.RD304.D7.C1,8,5.0,9
LAMI.RD304.D0.C1,9,11.0,6
LAMI.RD306.D11.C1,5,2.0,2
LAMI.RD308.D0.C2,24,19.0,22
LAMI.RD306.D14.C1,2,4.0,1
LAMI.RD304.D0.C3,10,6.0,8
LAMI.RD308.D0.C3,21,18.0,23
LAMI.RD306.D23.C1,13,13.0,15
LAMI.RD306.D14.C3,3,8.0,3


In [93]:
### uncomment to make 3 data type correlation figure

# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# from scipy.stats import pearsonr, norm
# import numpy as np

# # Extract columns
# x = combined_cutibacterium_df['V1-V3']
# y = combined_cutibacterium_df['V4']
# z = combined_cutibacterium_df['Shotgun']

# # Compute pairwise Pearson correlations with p-values
# r1, p1 = pearsonr(x, y)
# r2, p2 = pearsonr(x, z)
# r3, p3 = pearsonr(y, z)

# # Compute average r
# r_avg = np.mean([r1, r2, r3])

# # --- Estimate combined p-value using Fisher's Z transformation ---
# def fisher_z(r):
#     return np.arctanh(r)

# def inverse_fisher_z(z):
#     return np.tanh(z)

# # Convert to Fisher Z values
# z1 = fisher_z(r1)
# z2 = fisher_z(r2)
# z3 = fisher_z(r3)

# # Mean of z-scores
# z_avg = np.mean([z1, z2, z3])

# # Convert back to average r (already done)
# # r_avg = inverse_fisher_z(z_avg)

# # Standard error for Fisher z (n ≈ number of samples)
# n = len(x)
# se = 1 / np.sqrt(n - 3)

# # Compute z-statistic and p-value
# z_stat = z_avg / se
# p_avg = 2 * (1 - norm.cdf(abs(z_stat)))  # two-tailed

# # Create 3D scatter plot
# fig = plt.figure(figsize=(10, 6))  # Wider for spacing
# ax = fig.add_subplot(111, projection='3d')

# # Scatter points
# scatter = ax.scatter(x, y, z, c='orange', s=60, edgecolors='k')

# # Axes labels and title
# ax.set_xlabel('16S V1-V3 Rank')
# ax.set_ylabel('16S V4 Rank')
# ax.set_zlabel('Shotgun metaG Rank')
# ax.set_title('Correlation of Cutibacterium by Sequencing Method', fontsize=16)

# # Format correlation results with p-values
# textstr = '\n'.join((
#     f'V1-V3 vs. V4:',
#     f'    Pearson r = {r1:.2f}, p = {p1:.3g}',
#     '',
#     f'V1-V3 vs. Shotgun:',
#     f'    Pearson r = {r2:.2f}, p = {p2:.3g}',
#     '',
#     f'V4 vs. Shotgun:',
#     f'    Pearson r = {r3:.2f}, p = {p3:.3g}',
#     '',
#     f'All averaged:',
#     f'    Pearson r = {r_avg:.2f}, p = {p_avg:.3g}',
# ))

# # Annotate text, positioned further left
# plt.gcf().text(0.01, 0.35, textstr, fontsize=10)

# # Diagonal 1:1:1 correlation line
# min_val = min(x.min(), y.min(), z.min())
# max_val = max(x.max(), y.max(), z.max())

# ax.plot([min_val, max_val], [min_val, max_val], [min_val, max_val],
#         color='orange', linewidth=2, linestyle='-', label='1:1:1 Correlation Line')

# # Add n value (# of matched samples between all three data types)
# textstr = 'n = 22 matched samples'
# plt.gcf().text(0.01, 0.21, textstr, fontsize=10)


# # Draw dashed lines from each point to the 1:1:1 diagonal
# def project_to_diag(x, y, z):
#     direction = np.array([1, 1, 1]) / np.sqrt(3)
#     point = np.array([x, y, z])
#     proj_length = np.dot(point, direction)
#     return proj_length * direction

# for xi, yi, zi in zip(x, y, z):
#     x_proj, y_proj, z_proj = project_to_diag(xi, yi, zi)
#     ax.plot([xi, x_proj], [yi, y_proj], [zi, z_proj],
#             color='gray', linestyle='--', linewidth=0.7)

# # Add legend to label the orange line
# ax.legend(loc='lower left', bbox_to_anchor=(-0.4, 0.25))

# plt.tight_layout()
# plt.savefig('test_3d_with_all_corrs_and_avg_pval.png', dpi=600)


### Venn diagram comparing genera-level taxa between V1-V3 and V4

In [94]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=16)


# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')


# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{' g__Reyranella', ' g__Mycobacterium', ' g__Janibacter'}

Filtered Unique to V4 (>=10% prevalence):
{' g__Lactococcus', ' g__Pantoea', ' g__Chryseobacterium', ' g__Dolosigranulum', ' g__Massilia', ' g__Aeromonas', ' g__Jeotgalicoccus', ' g__Frederiksenia', ' g__Empedobacter', ' g__Brachybacterium', ' g__Moraxella', ' g__Turicella', ' g__Marinomonas', ' g__Gardnerella', ' g__Bradyrhizobium', ' g__Capnocytophaga', ' g__Leptotrichia', ' g__Finegoldia', ' g__Peptoniphilus', ' g__Atopobium', ' g__Abiotrophia', ' g__Fusobacterium', ' g__Blautia', ' g__Peptostreptococcus', ' g__Leuconostoc', ' g__Psychrobacter', ' g__Fenollaria', ' g__Luteimonas', ' g__Aggregatibacter', ' g__Vibrio', ' g__Aerococcus', ' g__Alloiococcus', ' g__Geobacillus', ' g__Bifidobacterium', ' g__Enhydrobacter', ' g__Stenotrophomonas'}

Filtered Shared Taxa (>=10% prevalence):
{' g__Lysobacter', ' g__Sphingopyxis', ' g__Brevundimonas', ' g__Cutibacterium', ' g__Brevibacterium',

### Phylogenetic tree

In [95]:
# Load the initial taxonomy file
taxonomy_file_path = '../Metadata/174116_taxonomy.tsv'
taxonomy_df = pd.read_csv(taxonomy_file_path, sep='\t')

# Extract the Genus from the Taxon column
taxonomy_df['Genus'] = taxonomy_df['Taxon'].str.extract(r'g__(\w+);?')

# Define the groups for each category
v1v3_unique = {'Janibacter', 'Reyranella', 'Mycobacterium'}
v4_unique = {'Empedobacter', 'Stenotrophomonas', 'Capnocytophaga', 'Aerococcus', 
             'Luteimonas', 'Aggregatibacter', 'Peptostreptococcus', 'Enhydrobacter', 
             'Abiotrophia', 'Bifidobacterium', 'Geobacillus', 'Psychrobacter', 'Massilia', 
             'Bradyrhizobium', 'Peptoniphilus', 'Aeromonas', 'Fusobacterium', 'Fenollaria', 
             'Dolosigranulum', 'Atopobium', 'Finegoldia', 'Chryseobacterium', 'Alloiococcus', 
             'Leptotrichia', 'Jeotgalicoccus', 'Gardnerella', 'Brachybacterium', 'Turicella', 
             'Frederiksenia', 'Lactococcus', 'Leuconostoc', 'Marinomonas', 'Pantoea', 
             'Moraxella', 'Vibrio', 'Blautia'
}
shared_taxa = {'Pseudomonas', 'Rothia', 'Gemella', 'Kocuria', 'Porphyromonas', 'Lysobacter', 
               'Nocardioides', 'Lactobacillus', 'Haemophilus', 'Thermus', 'Micrococcus', 
               'Actinomyces', 'Phenylobacterium', 'Neisseria', 'Paracoccus', 'Streptococcus', 
               'Brevibacterium', 'Veillonella', 'Brevundimonas', 'Anaerococcus', 'Caulobacter', 
               'Alloprevotella', 'Sphingopyxis', 'Corynebacterium', 'Prevotella', 'Staphylococcus', 
               'Lawsonella', 'Cutibacterium', 'uncultured', 'Limnobacter'

}

# Filter ASVs for the specified genera
filtered_df = taxonomy_df[taxonomy_df['Genus'].isin(v1v3_unique | v4_unique | shared_taxa)]

# Group by genus and get the most frequent ASV for each genus as the consensus
consensus_asvs = filtered_df.groupby('Genus')['Feature ID'].apply(lambda x: x.mode().iloc[0]).reset_index()
consensus_asvs.columns = ['Genus', 'Consensus ASV']

# Add group column based on the genus
consensus_asvs['group'] = consensus_asvs['Genus'].apply(
    lambda g: 'V1-V3 unique' if g in v1v3_unique else
              'V4 unique' if g in v4_unique else
              'Shared' if g in shared_taxa else
              'Unknown'
)

# Load the updated taxonomy file for resolving placeholders
taxonomy_df_updated = pd.read_csv(taxonomy_file_path, sep='\t')
taxonomy_df_updated['Genus'] = taxonomy_df_updated['Taxon'].str.extract(r'g__(\w+);?')

# Resolve placeholders using the taxonomy data
for index, row in consensus_asvs.iterrows():
    if row['Consensus ASV'].startswith('Placeholder_ASV'):
        genus = row['Genus']
        matching_asvs = taxonomy_df_updated[taxonomy_df_updated['Genus'] == genus]['Feature ID']
        if not matching_asvs.empty:
            consensus_asvs.at[index, 'Consensus ASV'] = matching_asvs.mode().iloc[0]

# Save the final DataFrame to a CSV file for review
consensus_asvs.to_csv('../Data/16S/primer_comparison/consensus_asvs.csv', index=False)


In [96]:
# Extract ASVs from the final_consensus_asvs DataFrame into FASTA format
output_fasta_path = "../Data/16S/primer_comparison/consensus_asvs.fasta"

with open(output_fasta_path, "w") as fasta_file:
    for index, row in consensus_asvs.iterrows():
        # Only include rows with valid ASVs (no placeholders)
        if not row['Consensus ASV'].startswith('Placeholder_ASV'):
            fasta_file.write(f">{row['Genus']}\n{row['Consensus ASV']}\n")

print(f"FASTA file created at: {output_fasta_path}")


FASTA file created at: ../Data/16S/primer_comparison/consensus_asvs.fasta
